# Time Series

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ixnix/GE5280PythonGeosciences/blob/main/colab/module_10/10_TimeSeries.ipynb)

*Run this notebook on Google Colab — no setup required.*

# Module 10: Time Series


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
import warnings
warnings.filterwarnings('ignore')

## Introduction

Time series data is essential for analyzing phenomena that change over time, such as weather, financial markets, or environmental measurements. In this module, you will learn how to handle dates and times in Python using the `datetime` module, work with time-indexed Pandas Series and DataFrames, and perform common operations such as resampling, aggregation, and slicing by time. By the end of this module, you will be able to organize, manipulate, and analyze temporal data using Pandas.

## Learning Objectives

- Understand and use Python's `datetime` module to work with dates and times.
- Work with time-indexed Series and DataFrames in Pandas for easy selection and slicing.
- Apply resampling techniques to aggregate or transform time series data at different frequencies.

## Reading
- [Chapter 11](https://wesmckinney.com/book/time-series)

## Datetime module

Time series data is an important form of structured data in many different fields, such as finance, economics, ecology, neuroscience, and physics. Anything that is observed or measured at many points in time forms a time series. Many time series are fixed frequency, which is to say that data points occur at regular intervals according to some rule, such as every 15 seconds, every 5 minutes, or once per month. Time series can also be irregular without a fixed unit of time or offset between units. How you mark and refer to time series data depends on the application, and you may have one of the following:

- Timestamps, specific instants in time
- Fixed periods, such as the month January 2007 or the full year 2010
- Intervals of time, indicated by a start and end timestamp. Periods can be thought of as special cases of intervals
- Experiment or elapsed time; each timestamp is a measure of time relative to a particular start time (e.g., the diameter of a cookie baking each second since being placed in the oven)

The Python standard library includes data types for date and time data, as well as calendar-related functionality. The **datetime**, **time**, and **calendar** modules are the main places to start.

1. Classes in datetime module

|Class| Description|
|----|------------|
|date| Store calendar date (year, month, day) using the Gregorian calendar|
|time| Store time of day as hours, minutes, seconds, and microseconds|
|datetime| Stores both date and time|
|timedelta| Represents the difference between two datetime values (as days, seconds, and microseconds)|
|tzinfo| Base type for storing time zone information|


In [ ]:
from datetime import datetime, timedelta, date

now = datetime.now()
now

In [ ]:
now.year, now.month, now.day, now.hour, now.minute, now.second

You can create a datetime object by specifying the year, month, day, hour, minute and/or second:

In [ ]:
date = datetime(year=2022,month=11,day=2,hour=12)
date

**timedelta** represents the temporal difference between two datetime objects:

In [ ]:
delta = datetime(2022,11,10) - datetime(2022,11,9,12)
delta

In [ ]:
from datetime import timedelta

t1 = datetime(2022,11,10)
t2 = t1 + timedelta(days=1,hours=3)
#t2 = t1 + timedelta(hours=27)

t2

You can format datetime objects and pandas Timestamp objects (introduced later) as strings using the **strftime()** method, passing a format specification.

There are two useful datetime methods to deal with conversion from string to datetime, and vice versa.

- **datetime.strftime(format)**: Return a string representing the date, controlled by an explicit format string. You can format datetime objects and pandas Timestamp objects (introduced later) as strings using the strftime method, passing a format specification
- **datetime.strptime(date_string, format)**: Return a datetime corresponding to date_string, parsed according to format.

List of datetime format specification

|Type| Description|
|----|------------|
|%Y| Four-digit year|
|%y| Two-digit year|
|%m| Two-digit month [01, 12]|
|%d| Two-digit day [01, 31]|
|%H| Hour (24-hour clock) [00, 23]|
|%I| Hour (12-hour clock) [01, 12]|
|%M| Two-digit minute [00, 59]|
|%S| Second [00, 61] (seconds 60, 61 account for leap seconds)|
|%w| Weekday as integer [0 (Sunday), 6]|
|%U| Week number of the year [00, 53]; Sunday is considered the first day of the week, and days before the first Sunday of the year are “week 0”|
|%W| Week number of the year [00, 53]; Monday is considered the first day of the week, and days before the first Monday of the year are “week 0”
|%z| UTC time zone offset as +HHMM or -HHMM; empty if time zone naive|
|%F| Shortcut for %Y-%m-%d (e.g., 2012-4-18)|
|%D| Shortcut for %m/%d/%y (e.g., 04/18/12)|
|%a| Weekday as locale’s abbreviated name (e.g., Sun, Mon, ...)|
|%A| Weekday as locale’s full name (e.g., Sunday, Monday, ...)|
|%b| Month as locale’s abbreviated name (e.g., Jan, Feb, ...|
|%B| Month as locale’s full name (e.g., January, Feburary, ...)|
|%p| Locale’s equivalent of either AM or PM|

In [ ]:
now = datetime.now()
now.strftime('Current time is %Y-%m-%d %H:%M:%S')

In [ ]:
now.strftime('Current time is %H:%M on %A, %B %d %Y')

In [ ]:
date_string = '2022 11-10 12:30 AM'
datetime.strptime(date_string,'%Y %m-%d %H:%M %p')

pandas is generally oriented toward working with arrays of dates, whether used as an axis index or a column in a DataFrame. The **to_datetime** method parses many different kinds of date representations.

In [ ]:
datestrs = ['11/6/2022 01:00:00', '11/6/2022 03:00:00', '11/6/2022 06:00:00']
pd.to_datetime(datestrs)


The **pd.date_range()** method is useful for generating a range of range of equally spaced time points (where the difference between any two adjacent points is specified by the given frequency).

Base time series frequencies:

|Alias| Offset type| Description|
|-----|------------|------------|
|D| Day| Calendar daily|
|B| BusinessDay| Business daily|
|H| Hour| Hourly|
|T or min| Minute| Minutely|
|S| Second| Secondly|
|L or ms| Milli| Millisecond (1/1,000 of 1 second)|
|U| Micro| Microsecond (1/1,000,000 of 1 second)|
|M| MonthEnd| Last calendar day of month|
|BM| BusinessMonthEnd| Last business day (weekday) of month|
|MS| MonthBegin| First calendar day of month|
|BMS| BusinessMonthBegin| First weekday of month|
|W-MON, W-TUE, ...| Week| Weekly on given day of week (MON, TUE, WED, THU, FRI, SAT, or SUN)|
|WOM-1MON, WOM-2MON, ...| WeekOfMonth| Generate weekly dates in the first, second, third, or fourth week of the month (e.g., WOM-3FRI for the third Friday of each month)|
|Q-JAN, Q-FEB, ...| QuarterEnd| Quarterly dates anchored on last calendar day of each month, for year ending in indicated month (JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, or DEC)|
|BQ-JAN, BQ-FEB, ...| BusinessQuarterEnd| Quarterly dates anchored on last weekday day of each month, for year ending in indicated month|
|QS-JAN, QS-FEB, ...| QuarterBegin| Quarterly dates anchored on first calendar day of each month, for year ending in indicated month|
|BQS-JAN, BQS-FEB, ...| BusinessQuarterBegin| Quarterly dates anchored on first weekday day of each month, for year ending in indicated month|
|A-JAN, A-FEB, ...| YearEnd| Annual dates anchored on last calendar day of given month (JAN, FEB, MAR, APR, MAY, JUN, JUL, AUG, SEP, OCT, NOV, or DEC)|
|BA-JAN, BA-FEB, ...| BusinessYearEnd| Annual dates anchored on last weekday of given month|
|AS-JAN, AS-FEB, ...| YearBegin| Annual dates anchored on first day of given month|
|BAS-JAN, BAS-FEB, ...| BusinessYearBegin| Annual dates anchored on first weekday of given month|

In [ ]:
pd.date_range(start='2022/1/1',end='2022/12/1',freq='M')

In [ ]:
pd.date_range(start='2022/1/1',end='2022/12/1',freq='MS')

In [ ]:
pd.date_range(start='2000',end='2023',freq='AS')

In [ ]:
pd.date_range(start='2022/11/10 00',periods=12,freq='h')

In [ ]:
pd.date_range(end='2022/11/10 00',periods=12,freq='h')

In [ ]:
pd.date_range(end='2022/11/10',periods=12,freq='3D')

## Resampling

**Resampling** refers to the process of converting a time series from one frequency to another.

Pandas objects are equipped with a resample method, which is the workhorse function for all frequency conversion. **resample()** has a similar API to **groupby()**; You call resample to group the data, then call an aggregation function.

Below we will use stock price data to show how resampling works.

In [ ]:
# use read_csv to read in historical stock price of Google, Inc.
# The first column is parsed as DatetimeIndex type and the index of the DataFrame by setting index_col=0.
goog = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_10/data/goog.csv',header=0,parse_dates=[0],index_col=0)
goog.sort_index(inplace=True)
goog

In [ ]:
goog.dtypes

Notice that the 'Close/Last', 'Open', 'High', and 'Low' columns are not float, and the numeric values have dollar signs before them. We need to convert the currency to float.

In [ ]:
goog['Close/Last'] = goog['Close/Last'].str.replace('$', '').astype('float')
goog['Open'] = goog['Open'].str.replace('$', '').astype('float')
goog['High'] = goog['High'].str.replace('$', '').astype('float')
goog['Low'] = goog['Low'].str.replace('$', '').astype('float')
goog['Volume'] = goog['Volume'].astype('float')
goog

In [ ]:
goog.dtypes

After sorting the index (by calling sort_index), the time series data is ordered chronologically. Then you can slice the DataFrame with timestamps not contained in a time series to perform a range query. You can pass either a string date, datetime, or timestamp. Remember that slicing in this manner produces views (shallow copy) on the source time series. 

In [ ]:
# Slice data for 2015
goog.loc['2015']

In [ ]:
# Slice data between 2015 and 2016
goog.loc['2015':'2016']

In [ ]:
# Slice data between two specific dates
goog.loc['2015-2-20':'2015-7-1']

In [ ]:
# Using a different date format
goog.loc['12/1/2015':'5/15/2016']

How do you calculate the monthly average of goog? 

We can use the **resampling** method. Resampling works in a similar way to groupby.

In [ ]:
# If you do not specify any columns, aggregation will be performed on all columns.
goog.resample('ME').mean()

In [ ]:
goog.resample('MS').mean()

In [ ]:
goog.resample('ME',kind='period').mean()

Similar to groupby, resample can work with the aggregation operation, and apply different functions to each column

In [ ]:
goog.resample('ME',kind='period').agg({'Close/Last':'mean','Volume':'sum'})

Resample is a flexible and high-performance method that can be used to process very large time series.

Resample method arguments:

|Argument| Description|
|--------|------------|
|freq| String or DateOffset indicating desired resampled frequency (e.g., ‘M', ’5min', or Second(15))|
|axis| Axis to resample on; default axis=0|
|fill_method| How to interpolate when upsampling, as in 'ffill' or 'bfill'; by default does no interpolation|
|closed| In downsampling, which end of each interval is closed (inclusive), 'right' or 'left'|
|label| In downsampling, how to label the aggregated result, with the 'right' or 'left' bin edge (e.g., the 9:30 to 9:35 five-minute interval could be labeled 9:30 or 9:35)|
|loffset| Time adjustment to the bin labels, such as '-1s' / Second(-1) to shift the aggregate labels one second earlier|
|limit| When forward or backward filling, the maximum number of periods to fill|
|kind| Aggregate to periods ('period') or timestamps ('timestamp'); defaults to the type of index the time series has|
|convention| When resampling periods, the convention ('start' or 'end') for converting the low-frequency period to high frequency; defaults to 'end'|


In [ ]:
goog['Close/Last'].plot(c='r',figsize=(5,3),label='Original')
goog['Close/Last'].resample('ME',kind='period').mean().plot(c='b',label='Monthly')
plt.legend()

In [ ]:
# read in the historical stock price of Apple, Inc.
aapl = pd.read_csv('https://raw.githubusercontent.com/ixnix/GE5280PythonGeosciences/main/module_10/data/aapl.csv',header=0,parse_dates=[0],index_col=0)
aapl.sort_index(inplace=True)
aapl['Close/Last'] = aapl['Close/Last'].str.replace('$', '').astype('float')
aapl['Open'] = aapl['Open'].str.replace('$', '').astype('float')
aapl['High'] = aapl['High'].str.replace('$', '').astype('float')
aapl['Low'] = aapl['Low'].str.replace('$', '').astype('float')
aapl['Volume'] = aapl['Volume'].astype('float')
aapl

In [ ]:
# Concatenate the Close/Last columns from two DataFrames, and rename them to 'goog' and 'aapl'
dfstock = pd.concat([goog['Close/Last'], aapl['Close/Last']], axis=1,keys=['goog','aapl'])
dfstock

In [ ]:
dfstock.plot(figsize=(5,3))

In [ ]:
# The above plot does not show the co-variations of goog and appl, due to large difference in the magnitude.
# We can normalize the two time series by their respective mean and standard deviation.
dfstock = (dfstock-dfstock.mean())/dfstock.std()
dfstock.plot(figsize=(5,3))

In [ ]:
# We confirm that the correlation between the goog and appl stock prices are very high (r=0.95)
dfstock.corr()